In [ ]:
#2

# Install libraries
!pip install transformers torch nltk

from transformers import pipeline
import nltk
from nltk.tokenize import word_tokenize

# Download NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

# Load transformer model
sentiment_pipeline = pipeline("sentiment-analysis")

# NLTK function
def nltk_process(text):
    return word_tokenize(text.lower())

while True:
    text = input("Enter a sentence (or type 'exit'): ")

    if text.lower() == "exit":
        break

    _ = nltk_process(text)

    # Transformer prediction
    result = sentiment_pipeline(text)[0]

    # output
    print("Sentiment:", result['label'])
    print("Confidence:", round(result['score'], 3))

In [ ]:
#3

# Install required libraries

!pip install pandas nltk prefixspan


# Imports

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from prefixspan import PrefixSpan

# =========================
# Download NLTK resources
# =========================

nltk.download('punkt')
nltk.download('punkt_tab')

print("\nLoading Dataset...\n")

# =========================
# Load dataset
# =========================
df = pd.read_csv("clinical_events.csv")

print("Dataset Preview:\n")
print(df.head())

# =========================
# NLTK preprocessing
# =========================
def preprocess_event(event):
    tokens = word_tokenize(str(event).lower())
    return tokens[0] if tokens else ""

df["processed_event"] = df["event"].apply(preprocess_event)

# =========================
# Create patient sequences
# =========================
sequences = []

for patient_id, group in df.groupby("patient_id"):
    ordered = group.sort_values("time")
    sequence = ordered["processed_event"].tolist()
    sequences.append(sequence)

print("\nPatient Event Sequences:\n")

for i, seq in enumerate(sequences, start=1):
    print("Patient", i, ":", seq)

# =========================
# Sequential pattern mining
# =========================
print("\nFrequent Clinical Pathways\n")

ps = PrefixSpan(sequences)

patterns = ps.frequent(2)

important_pathways = []

for support, pattern in patterns:

    # show meaningful pathways (length >= 3)
    if len(pattern) >= 3:
        pathway = " → ".join(pattern)

        print("Patients:", support)
        print("Pathway :", pathway)
        print()

        # check important pathway condition
        if pattern[0] == "admission" and pattern[-1] == "discharge":
            important_pathways.append((support, pattern))

# =========================
# Important pathways section
# =========================
print("\n==============================")
print("IMPORTANT CLINICAL PATHWAYS")
print("==============================\n")

for support, pattern in important_pathways:
    pathway = " → ".join(pattern)

    print("Patients:", support)
    print("Pathway :", pathway)
    print()

In [ ]:
#1


# Install libraries
#!pip install spacy nltk networkx matplotlib

# Download spaCy model
!python -m spacy download en_core_web_sm

import nltk
from nltk.tokenize import word_tokenize
import spacy
import networkx as nx
import matplotlib.pyplot as plt

# NLTK (name sake usage)
nltk.download('punkt')
nltk.download('punkt_tab')

# Load spaCy model
nlp = spacy.load("en_core_web_sm")


# -------------------------------
#  TRIPLE EXTRACTION
# -------------------------------
def extract_triples(text):
    doc = nlp(text)
    triples = []

    for token in doc:

        # -------- Rule 1: Normal verbs --------
        if token.pos_ == "VERB":
            subject = None
            obj = None

            for child in token.children:
                if child.dep_ in ("nsubj", "nsubjpass"):
                    subject = child.text
                if child.dep_ in ("dobj", "attr", "pobj"):
                    obj = child.text

            if subject and obj:
                triples.append((subject, token.lemma_, obj))

        # -------- Rule 2: "is/am/are" --------
        if token.dep_ == "ROOT" and token.lemma_ == "be":
            subject = None
            complement = None

            for child in token.children:
                if child.dep_ == "nsubj":
                    subject = child.text
                if child.dep_ in ("attr", "acomp"):
                    complement = child.text

            if subject and complement:
                triples.append((subject, "is", complement))

    return list(set(triples))


# -------------------------------
# GRAPH VISUALIZATION
# -------------------------------
def draw_graph(triples):
    G = nx.DiGraph()

    for subj, rel, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj, label=rel)

    pos = nx.spring_layout(G)

    plt.figure()
    nx.draw(G, pos, with_labels=True)

    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    plt.show()


# -------------------------------
# MAIN LOOP
# -------------------------------
while True:
    text = input("Enter a sentence (or type 'exit'): ")

    if text.lower() == "exit":
        break

    # NLTK
    _ = word_tokenize(text)

    triples = extract_triples(text)

    print("\nTriples:", triples)

    if triples:
        draw_graph(triples)
    else:
        print("No relationships found")

    print("-" * 40)

In [ ]:
#4

# Install (Colab only)
#!pip install nltk --quiet

import pandas as pd
import nltk
import string
import warnings
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")
nltk.download('punkt')
nltk.download('stopwords')

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Preg","Glucose","BP","Skin","Insulin","BMI","DPF","Age","Outcome"]
df = pd.read_csv(url, names=cols)

def symptoms(r):
    s=[]
    if r["Glucose"]>140: s.append("high sugar")
    if r["BMI"]>30: s.append("obesity")
    if r["BP"]>90: s.append("bp")
    if r["Age"]>50: s.append("fatigue")
    return " ".join(s)

stop = set(stopwords.words("english"))
def clean(t):
    return " ".join([w for w in word_tokenize(t.lower()) if w not in stop and w not in string.punctuation])

df["Symptoms"] = df.apply(symptoms, axis=1).apply(clean)
df["G_BMI"] = df["Glucose"]*df["BMI"]

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

num = ["Preg","Glucose","BP","Skin","Insulin","BMI","DPF","Age","G_BMI"]

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num),
        ("txt", TfidfVectorizer(), "Symptoms")
    ])),
    ("model", LogisticRegression(max_iter=3000, solver="liblinear"))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(Xtr, ytr)

pred = pipe.predict(Xte)
prob = pipe.predict_proba(Xte)[:,1]

print("Accuracy:", round(accuracy_score(yte, pred),4))
print("ROC-AUC:", round(roc_auc_score(yte, prob),4))
print("\nClassification Report:\n", classification_report(yte, pred))

print("Confusion Matrix:\n", confusion_matrix(yte, pred))

cv = cross_val_score(pipe, X, y, cv=5)
print("CV Accuracy:", round(cv.mean(),4))



In [ ]:
# install
#!pip install pandas matplotlib seaborn scikit-learn nltk

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

# upload dataset
from google.colab import files
uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_csv(file).dropna()
print("Records:", len(df))

# preprocessing
sw = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in sw and len(w) > 2]
    return " ".join(tokens)

df['clean'] = df['text'].apply(clean_text)

# split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'],
    test_size=0.3,
    random_state=42,
    stratify=df['label']
)

# vectorization
vec = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train = vec.fit_transform(X_train)
X_test = vec.transform(X_test)

# model
model = LinearSVC()
model.fit(X_train, y_train)

# prediction
y_pred = model.predict(X_test)

# results
print("\nAccuracy:", round(accuracy_score(y_test, y_pred)*100,2), "%")
print("\n", classification_report(y_test, y_pred))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)


plt.figure(figsize=(12,5))

# Real vs Fake
plt.subplot(1,2,1)
sns.countplot(x=y_test)
plt.title("Real vs Fake (Actual)")
plt.xticks([0,1], ['Real','Fake'])
plt.xlabel("Class")
plt.ylabel("Count")

# Confusion Matrix
plt.subplot(1,2,2)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Real','Fake'],
            yticklabels=['Real','Fake'],
            cmap='Blues')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
import nbformat

with open('/content/Nlp.ipynb', 'r', encoding='utf-8') as f:

    nb = nbformat.read(f, as_version=4)

if "widgets" in nb["metadata"]:

    del nb["metadata"]["widgets"]

with open('/content/Nlp_clean.ipynb', 'w', encoding='utf-8') as f:

    nbformat.write(nb, f)

print("✅ Cleaned notebook saved!")